# DIMBA → Shakespeare (full works, ~30M, Mamba2 / dual-T4)

Pretrains a ~30M-param latent-diffusion Mamba model **unconditionally** on the
complete works of Shakespeare, using the precompiled `mamba_ssm` CUDA kernels
across both T4 GPUs (DDP).

**Run order:** 1 (check) → 3 (setup+data) → 5 (train) → 7 (sample) → 9 (upload).
Set **Accelerator = GPU T4 x2** and add the `dimba-lib-exp` dataset as input.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}')

## 1. Setup — copy code, install kernels, download + clean the corpus

In [ ]:
import os, sys, shutil, urllib.request, unicodedata

# Locate the uploaded dataset (contains the repo) and copy it into /kaggle/working.
INPUT_SRC = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train_shakespeare.py' in files:
        INPUT_SRC = os.path.dirname(root)
        break
if INPUT_SRC is None:
    raise FileNotFoundError('Dataset not found! Add the dimba-lib-exp dataset as input.')
print(f'Found: {INPUT_SRC}')

REPO = '/kaggle/working/dimba-lib-exp'
if os.path.exists(REPO):
    shutil.rmtree(REPO)
shutil.copytree(INPUT_SRC, REPO)
os.chdir(REPO)
sys.path.insert(0, 'src')

!pip install -q -e .
!pip install -q tensorboard huggingface_hub
# Mamba SSD CUDA kernels (fast, low-memory). May take a few min on first build.
!pip install -q --no-build-isolation causal-conv1d mamba-ssm
try:
    import mamba_ssm
    print('OK mamba_ssm', getattr(mamba_ssm, '__version__', '?'))
except Exception as e:
    print('WARNING mamba_ssm import failed:', e)

# --- Data: complete works of Shakespeare (Project Gutenberg #100), cleaned ---
os.makedirs('data', exist_ok=True)
req = urllib.request.Request('https://www.gutenberg.org/cache/epub/100/pg100.txt',
                             headers={'User-Agent': 'Mozilla/5.0'})
raw = urllib.request.urlopen(req).read().decode('utf-8')

# Strip the Gutenberg license header/footer.
s = raw.find('*** START OF THE PROJECT GUTENBERG EBOOK')
s = raw.find('\n', s) + 1
e = raw.find('*** END OF THE PROJECT GUTENBERG EBOOK')
text = raw[s:e]

# Normalize smart punctuation, then drop any remaining non-ASCII (newlines kept).
repl = {'\u2018':"'", '\u2019':"'", '\u201c':'"', '\u201d':'"', '\u2014':'--',
        '\u2013':'-', '\u2026':'...', '\u2022':'*', '\u2122':'(TM)', '\ufeff':''}
for k, v in repl.items():
    text = text.replace(k, v)
text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('ascii')

with open('data/shakespeare.txt', 'w') as f:
    f.write(text)
print(f'Repo: {REPO}')
print(f'Data: {len(text):,} chars (cleaned complete works)  |  Ready.')

## 2. Train — ~30M params, both T4s, fp32, cosine LR, early-stop safety net

Uses **fp32** (`32-true`) for numerical stability (fp16 overflowed at this depth/length).
First training step takes ~2-4 min while Triton compiles the kernel — **not stuck**.
Watch for `ssm kernel : Mamba2` and `train/loss` trending down and **staying finite**.

In [ ]:
!python scripts/train_shakespeare.py \
    --d-model 768 --num-layers 12 --d-state 128 \
    --seq-len 256 --num-steps 128 \
    --use-mamba-ssm \
    --epochs 30 --batch-size 64 \
    --lr 3e-4 --warmup-steps 300 --patience 8 \
    --num-workers 2 --grad-clip 1.0 \
    --accelerator gpu --devices 2 --strategy ddp_find_unused_parameters_true \
    --precision 32-true

## 3. Sample — unconditional generation from the best checkpoint

In [ ]:
import torch
from dimba.training import DIMBALightningModule
from dimba.tokenizers import SimpleCharacterTokenizer
from dimba.diffusion.sampling import sample_from_model

# Rebuild the exact model from the saved hyperparameters, then load weights.
module = DIMBALightningModule.load_from_checkpoint(
    'checkpoints/shakespeare/shakespeare1.ckpt', map_location='cuda')
model = module.model.cuda().eval()

cfg        = dict(module.hparams['model_config'])
n_params   = sum(p.numel() for p in model.parameters())
outer_d    = cfg['d_model']
d_latent   = cfg.get('d_latent', outer_d)
num_layers = cfg['num_denoiser_layers']
print(f'{n_params:,} params | d_model={outer_d} latent={d_latent} layers={num_layers}')
print('ssm kernel :', type(model.denoiser.blocks[0].mamba_fwd).__name__)

# Use the exact tokenizer saved during training.
tok = SimpleCharacterTokenizer(vocab_size=128)
tok.load('checkpoints/shakespeare/tokenizer.json')

# Unconditional generation (the model is pretrained without prompts).
for k in range(4):
    with torch.no_grad():
        gen = sample_from_model(model, prompt_ids=None, seq_len=400,
                                num_steps=50, temperature=0.8, top_p=0.95, device='cuda')
    print(f'\n========== sample {k + 1} ==========')
    print(tok.decode(gen[0]))

## 4. (Optional) Upload to Hugging Face — needs `HF_TOKEN` in Kaggle Secrets

In [ ]:
from huggingface_hub import HfApi, create_repo
from datetime import datetime

token = os.environ.get('HF_TOKEN')
if not token:
    print('ERROR: HF_TOKEN not set. Kaggle -> Add-ons -> Secrets -> add HF_TOKEN')
else:
    api = HfApi(token=token)
    username = api.whoami()['name']
    repo_id = f'{username}/dimba-shakespeare'
    create_repo(repo_id, private=True, token=token, exist_ok=True)
    for fn in ['shakespeare1.ckpt', 'tokenizer.json']:
        api.upload_file(path_or_fileobj=f'checkpoints/shakespeare/{fn}',
                        path_in_repo=fn, repo_id=repo_id, token=token)
    card = f"""---
license: mit
tags: [dimba, diffusion, mamba, shakespeare]
---
# DIMBA Shakespeare
Latent-diffusion Mamba pretrained (unconditionally) on the complete works of Shakespeare.
- Params: {n_params:,}
- Config: d_model={outer_d}, latent={d_latent}, layers={num_layers}
- Trained: {datetime.now().strftime("%Y-%m-%d")}
"""
    api.upload_file(path_or_fileobj=card.encode(), path_in_repo='README.md',
                    repo_id=repo_id, token=token)
    print(f'https://huggingface.co/{repo_id} (private)')